In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score

In [6]:
train_df = pd.read_csv("stage_1_regression_train.csv")
test_df = pd.read_csv("stage_1_regression_test_features.csv")

In [7]:
def prepare(df):
  df = df.copy()

  rh = df["relative_humidity_2m"].clip(0.01,100)
  dp = df["dew_point_2m"]

  #ЛИН ПРИБЛЕЖЕНИЕ МАГНУСА КАРЛСЕНА ДЛЯ ТЕМПРЭЧУР
  df["temp_magnus_simple"] = dp+(100.0-rh)/5.0

  a= 17.27
  b=237.7
  alpha=((a*dp)/(b*dp))+np.log(rh/100.0)
  df["temp_magnus_exact"] = (b*alpha)/(a-alpha)

  wind_rad = np.radians(df["wind_direction_10m"])
  df["wind_x"] = df["wind_speed_10m"] * np.cos(wind_rad)
  df["wind_y"] = df["wind_speed_10m"] * np.sin(wind_rad)

  return df



In [8]:
train_feat = prepare(train_df)
test_feat = prepare(test_df)

In [9]:
train_feat.shape

(4089, 14)

In [10]:
test_feat.shape

(1023, 14)

In [11]:
train_feat

,target,relative_humidity_2m,dew_point_2m,cloud_cover,wind_speed_10m,wind_direction_10m,uv_index,is_day,sunshine_duration,shortwave_radiation,temp_magnus_simple,temp_magnus_exact,wind_x,wind_y
0,13.0040,79.0,9.446486,100.0,8.905908,345.963700,1.90,1.0,699.35077,332.0,13.646486,-2.223430,8.639998,-2.160008
1,16.2040,62.0,8.914721,100.0,10.464798,3.945108,0.25,1.0,0.00000,73.0,16.514721,-5.451600,10.440001,0.719986
2,13.5040,82.0,10.490725,50.0,3.396233,327.994660,0.00,0.0,0.00000,0.0,14.090725,-1.718909,2.880001,-1.799998
3,11.8540,95.0,11.078381,24.0,4.072935,135.000100,0.00,0.0,0.00000,0.0,12.078381,0.294376,-2.880005,2.879995
4,20.3040,60.0,12.285676,0.0,7.594208,328.570500,0.40,1.0,3600.00000,163.0,20.285676,-5.881649,6.480004,-3.959993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4084,19.0040,51.0,8.637112,97.0,12.889810,35.909817,1.75,1.0,3600.00000,378.0,18.437112,-7.989843,10.439988,7.560017
4085,21.9535,71.0,16.455479,100.0,9.346143,164.357700,3.30,1.0,3144.71100,323.0,22.255479,-3.656816,-8.999997,2.520008
4086,1.3035,91.0,-0.003951,87.0,2.545584,261.870000,0.00,0.0,0.00000,0.0,1.796049,-0.297696,-0.359995,-2.520000
4087,17.1540,76.0,12.887712,100.0,7.289445,212.905240,2.90,1.0,3600.00000,430.0,17.687712,-2.745206,-6.120001,-3.960000


In [12]:
test_idx = test_feat["index"]
X_test = test_feat.drop(columns=["index"])
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

In [14]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=0)

обучение rfr

In [17]:
X = train_feat.drop(columns=["target"])
y = train_feat['target']

In [15]:
kf

KFold(n_splits=5, random_state=0, shuffle=True)

In [18]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X,y),1):
  X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
  X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

  rf = RandomForestRegressor(
      n_estimators=200,
      max_depth=18, # 15..20 - нет переобучения
      min_samples_split=2,
      min_samples_leaf=1,
      random_state=0+fold,
      n_jobs=-1, # ядра процессор
  )

  rf.fit(X_tr,y_tr)

  oof_preds[val_idx]=rf.predict(X_va)
  test_preds+=rf.predict(X_test)/kf.n_splits

In [19]:
cv_rmse = np.sqrt(mean_squared_error(y,oof_preds))
print(f"kf rmse: {cv_rmse:.6f}")

kf rmse: 0.152633


In [20]:
score = max(0,min(100,10+100*(0.65787-cv_rmse)/0.65787))
print(f"{score:.2f}")
print(86.05/100)

86.80
0.8604999999999999
